# 08 — YAMNet Transfer Learning

Use a pretrained YAMNet (AudioSet) model to extract 1024-d embeddings from 1 s, 16 kHz mono WAV segments (from Notebook 02), then train a lightweight classification head (Dense layers) for the 3 classes (angle_grinder, background, tools). Optionally fine-tune with a lower learning rate. Saves embeddings, trained head, metrics, and plots.


In [9]:
# Imports and setup
from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf
import tensorflow_hub as hub  # Correct: no 'from tensorflow_hub import tf_keras'
import soundfile as sf
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import yaml, json, os
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
UNIV_DIR = PROJECT_ROOT / 'data' / 'processed' / 'universal'
EMB_DIR = PROJECT_ROOT / 'data' / 'processed' / 'neural_features'
EMB_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
for d in [METRICS_DIR, FIG_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('TF:', tf.__version__)
print('TF-Hub:', hub.__version__)

# Verify tf-keras is available (optional diagnostic)
try:
    import tf_keras
    print('tf-keras version:', tf_keras.__version__)
except ImportError:
    print('Warning: tf-keras not found. Install with: pip install tf-keras')


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4
TF: 2.14.0
TF-Hub: 0.16.1
tf-keras version: 2.15.0


In [10]:
# Load config
with open(CFG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
audio_cfg = cfg.get('audio', {})
TARGET_SR = int(audio_cfg.get('sample_rate', 16000))
CLASSES = ['angle_grinder', 'background', 'tools']
class_to_idx = {c:i for i,c in enumerate(CLASSES)}
print('Using classes:', class_to_idx)


Using classes: {'angle_grinder': 0, 'background': 1, 'tools': 2}


In [11]:
# Gather segment file paths and labels using the preprocessing manifest
manifest_path = METRICS_DIR / 'preprocessing_manifest.csv'
if not manifest_path.exists():
    raise FileNotFoundError('preprocessing_manifest.csv not found — run Notebook 02 first')
man = pd.read_csv(manifest_path)
# Ensure required columns exist
assert 'segment_path' in man.columns and 'label' in man.columns, 'Manifest missing segment_path/label columns'
# Keep only files that currently exist on disk
man = man[man['segment_path'].apply(lambda p: Path(str(p)).exists())].reset_index(drop=True)
print('Total segments for YAMNet:', len(man))
# Build arrays of paths and integer labels
paths = man['segment_path'].astype(str).tolist()
labels = man['label'].map(class_to_idx).astype(int).values
print('Label distribution:', np.bincount(labels, minlength=len(CLASSES)))


Total segments for YAMNet: 19124
Label distribution: [6458 8069 4597]


In [12]:
# Load YAMNet from TF Hub (expects mono waveform at 16 kHz)
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print('YAMNet loaded.')

def load_waveform_16k(path):
    # Load with soundfile to float32, ensure 16 kHz
    y, sr = sf.read(path, dtype='float32')
    if y.ndim > 1:
        y = librosa.to_mono(y.T)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR, res_type='soxr_hq')
    return y

def compute_embedding(path):
    wav = load_waveform_16k(path)
    # YAMNet expects shape [samples], float32 in [-1,1]
    waveform = tf.convert_to_tensor(wav, dtype=tf.float32)
    scores, embeddings, spectrogram = yamnet_model(waveform)
    # Average embeddings over time frames to get a 1024-d vector
    emb = tf.reduce_mean(embeddings, axis=0).numpy()
    return emb

# Compute or load cached embeddings
emb_cache = EMB_DIR / 'yamnet_embeddings.npy'
lab_cache = EMB_DIR / 'yamnet_labels.npy'
if emb_cache.exists() and lab_cache.exists():
    X = np.load(emb_cache)
    y = np.load(lab_cache)
    print('Loaded cached embeddings:', X.shape)
else:
    X_list = []
    for i, p in enumerate(paths):
        try:
            X_list.append(compute_embedding(p))
        except Exception as e:
            # Skip problematic files
            if i % 100 == 0:
                print(f'Error on {p}:', e)
            continue
    # Align y to successful X count
    n = len(X_list)
    X = np.stack(X_list, axis=0) if n else np.empty((0,1024), dtype=np.float32)
    y = labels[:n]
    np.save(emb_cache, X); np.save(lab_cache, y)
    print('Saved embeddings:', X.shape)

num_classes = len(CLASSES)
print('Embeddings shape:', X.shape, 'Labels:', y.shape)


YAMNet loaded.
Saved embeddings: (19124, 1024)
Embeddings shape: (19124, 1024) Labels: (19124,)


In [13]:
# File-level split if possible using manifest stems
if 'original_path' in man.columns:
    stems = man['original_path'].apply(lambda p: Path(str(p)).stem).values[:len(y)]
    df_idx = pd.DataFrame({'idx': np.arange(len(y)), 'y': y, 'stem': stems})
    uniq = df_idx[['stem','y']].drop_duplicates()
    tr, te = train_test_split(uniq, test_size=0.2, stratify=uniq['y'], random_state=42)
    tr_stems, te_stems = set(tr['stem']), set(te['stem'])
    tr_idx = df_idx['idx'][df_idx['stem'].isin(tr_stems)].values
    te_idx = df_idx['idx'][df_idx['stem'].isin(te_stems)].values
    X_train, X_test, y_train, y_test = X[tr_idx], X[te_idx], y[tr_idx], y[te_idx]
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
else:
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
print('Splits:', X_train.shape, X_val.shape, X_test.shape)
# Class weights for imbalance
cnt = np.bincount(y_train, minlength=num_classes)
class_weight = {i: float(cnt.max()/c) for i,c in enumerate(cnt) if c>0}
print('Class weights:', class_weight)


Splits: (12720, 1024) (3180, 1024) (3224, 1024)
Class weights: {0: 1.175145024542615, 1: 1.0, 2: 1.7728037697744867}


In [17]:
# Build classification head (1024 -> 128 -> 64 -> num_classes)
from tensorflow import keras
from tensorflow.keras import layers

def make_head(input_dim=1024, num_classes=3, dropout1=0.4, dropout2=0.3):
    inputs = keras.Input(shape=(input_dim,), name='emb')
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.Dropout(dropout1)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='yamnet_head')
    return model

head = make_head(1024, num_classes)
head.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
head.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint(
        filepath=str(PROJECT_ROOT / 'models' / 'neural' / 'yamnet_head_best.weights.h5'),
        save_best_only=True,
        save_weights_only=True,           # <- avoids model.save() during training
        monitor='val_accuracy'
    ),
]

hist = head.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=128,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

# Optionally reload best weights before final save
head.load_weights(str(PROJECT_ROOT / 'models' / 'neural' / 'yamnet_head_best.weights.h5'))

# Save final full model in the native Keras format
head.save(str(PROJECT_ROOT / 'models' / 'neural' / 'yamnet_head_final.keras'))


Model: "yamnet_head"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 emb (InputLayer)            [(None, 1024)]            0         
                                                                 
 dense_9 (Dense)             (None, 128)               131200    
                                                                 
 dropout_6 (Dropout)         (None, 128)               0         
                                                                 
 dense_10 (Dense)            (None, 64)                8256      
                                                                 
 dropout_7 (Dropout)         (None, 64)                0         
                                                                 
 dense_11 (Dense)            (None, 3)                 195       
                                                                 
Total params: 139651 (545.51 KB)
Trainable params: 1396

2025-11-15 10:39:45.552883: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


100/100 [==============================] - 1s 8ms/step - loss: 0.8714 - accuracy: 0.7524 - val_loss: 0.4027 - val_accuracy: 0.8579 - lr: 0.0010
Epoch 2/15
100/100 [==============================] - 0s 5ms/step - loss: 0.6778 - accuracy: 0.8406 - val_loss: 0.3680 - val_accuracy: 0.8695 - lr: 0.0010
Epoch 3/15
100/100 [==============================] - 0s 5ms/step - loss: 0.6328 - accuracy: 0.8598 - val_loss: 0.3336 - val_accuracy: 0.8893 - lr: 0.0010
Epoch 4/15
100/100 [==============================] - 0s 5ms/step - loss: 0.6225 - accuracy: 0.8680 - val_loss: 0.3283 - val_accuracy: 0.8987 - lr: 0.0010
Epoch 5/15
100/100 [==============================] - 0s 5ms/step - loss: 0.6541 - accuracy: 0.8719 - val_loss: 0.3010 - val_accuracy: 0.8991 - lr: 0.0010
Epoch 6/15
100/100 [==============================] - 0s 5ms/step - loss: 0.6367 - accuracy: 0.8708 - val_loss: 0.3136 - val_accuracy: 0.9085 - lr: 0.0010
Epoch 7/15
100/100 [==============================] - 0s 5ms/step - loss: 0.6046 

In [19]:
try:
    yamnet_layer = hub.KerasLayer('https://tfhub.dev/google/yamnet/1', trainable=False)
    # Build end-to-end model
    inp = keras.Input(shape=(None,), dtype=tf.float32, name='waveform')
    
    # YAMNet outputs a dict or single tensor; access embeddings explicitly
    yamnet_outputs = yamnet_layer(inp)
    # Try accessing by key (if dict) or index (if tuple)
    if isinstance(yamnet_outputs, dict):
        embeddings = yamnet_outputs['embeddings']  # TF-Hub models often return dicts
    else:
        # If tuple, embeddings is typically index 1: (scores, embeddings, spectrogram)
        embeddings = yamnet_outputs[1] if len(yamnet_outputs) > 1 else yamnet_outputs
    
    x = tf.reduce_mean(embeddings, axis=1)  # (batch, 1024)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    ft_model = keras.Model(inp, out, name='yamnet_end2end')
    ft_model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print('End-to-end model constructed (frozen YAMNet).')
except Exception as e:
    print('Fine-tune graph construction skipped:', e)



Fine-tune graph construction skipped: Exception encountered when calling layer "keras_layer_1" (type KerasLayer).

in user code:

    File "/Users/harryirving/Development/projects/ai-ml/BikeAIv4/venv/lib/python3.10/site-packages/tensorflow_hub/keras_layer.py", line 242, in call  *
        result = f()

    TypeError: Binding inputs to tf.function failed due to `Can not cast TensorSpec(shape=(None, None), dtype=tf.float32, name=None) to TensorSpec(shape=(None,), dtype=tf.float32, name=None)`. Received args: (<tf.Tensor 'Placeholder:0' shape=(None, None) dtype=float32>,) and kwargs: {} for signature: (waveform: TensorSpec(shape=(None,), dtype=tf.float32, name=None)).


Call arguments received by layer "keras_layer_1" (type KerasLayer):
  • inputs=tf.Tensor(shape=(None, None), dtype=float32)
  • training=None


In [20]:
# Evaluation on test set (head)
y_pred = head.predict(X_test).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=CLASSES))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('YAMNet Head — Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Pred')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cm_yamnet_head.png', dpi=150)
plt.close()
print('Saved confusion matrix.')


101/101 [==============================] - 0s 1ms/step
               precision    recall  f1-score   support

angle_grinder       0.79      0.87      0.83       855
   background       0.96      0.95      0.96      1485
        tools       0.81      0.76      0.78       884

     accuracy                           0.88      3224
    macro avg       0.86      0.86      0.86      3224
 weighted avg       0.88      0.88      0.88      3224

Saved confusion matrix.
